In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Parameters ---
TICKER = "RKLB"
PERIOD = "2y"        # 2 years of historical data

# --- Fetch closing prices ---
data = yf.download(TICKER, period=PERIOD, auto_adjust=True)
prices = data["Close"].squeeze()

# --- Calculate daily log-returns ---
log_returns = np.log(prices / prices.shift(1)).dropna()

# --- Estimate drift (mu) and volatility (sigma) ---
mu = log_returns.mean()
sigma = log_returns.std()

print(f"Ticker:     {TICKER}")
print(f"Daily mu:   {mu:.6f}")
print(f"Daily sigma:{sigma:.6f}")
print(f"Ann. vol:   {sigma * np.sqrt(252):.2%}")

# Quick look at the return distribution
log_returns.hist(bins=60, figsize=(8, 4), color="steelblue", edgecolor="white")
plt.title(f"{TICKER} daily log-returns")
plt.xlabel("Log-return")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# --- Parameters ---
N_SIMULATIONS = 1000   # number of price paths
N_DAYS = 252           # trading days to simulate (1 year)
S0 = prices.iloc[-1]   # starting price (most recent closing price)

# --- Simulate paths using Geometric Brownian Motion ---
np.random.seed(42)     # makes results reproducible

# Generate random daily shocks for all simulations at once
random_shocks = np.random.normal(0, 1, (N_DAYS, N_SIMULATIONS))

# GBM formula: drift term + random volatility term
daily_returns = (mu - 0.5 * sigma**2) * 1 + sigma * random_shocks

# Convert daily returns into price paths
price_paths = S0 * np.exp(np.cumsum(daily_returns, axis=0))

print(f"Starting price:  ${S0:.2f}")
print(f"Simulations:     {N_SIMULATIONS}")
print(f"Trading days:    {N_DAYS}")
print(f"Median end price: ${np.median(price_paths[-1]):.2f}")
print(f"Mean end price:   ${np.mean(price_paths[-1]):.2f}")

In [ ]:
!pip install plotly

In [ ]:
import plotly.graph_objects as go

# --- Colour paths by final outcome (bottom 10%, middle 80%, top 10%) ---
final_prices = price_paths[-1]
low_threshold = np.percentile(final_prices, 10)
high_threshold = np.percentile(final_prices, 90)

fig = go.Figure()

# --- Plot all 1000 paths ---
for i in range(N_SIMULATIONS):
    final = final_prices[i]
    if final < low_threshold:
        color = "rgba(220, 53, 53, 0.12)"    # red — bottom 10%
    elif final > high_threshold:
        color = "rgba(40, 167, 69, 0.12)"    # green — top 10%
    else:
        color = "rgba(100, 149, 237, 0.08)"  # blue — middle 80%

    fig.add_trace(go.Scatter(
        y=price_paths[:, i],
        mode="lines",
        line=dict(color=color, width=0.8),
        hovertemplate="Day %{x}<br>Price: $%{y:.2f}<extra></extra>",
        showlegend=False
    ))

# --- Median path on top ---
median_path = np.median(price_paths, axis=1)
fig.add_trace(go.Scatter(
    y=median_path,
    mode="lines",
    line=dict(color="white", width=2.5, dash="dash"),
    name="Median path",
    hovertemplate="Day %{x}<br>Median: $%{y:.2f}<extra></extra>"
))

# --- Starting price line ---
fig.add_hline(
    y=S0,
    line_dash="dot",
    line_color="yellow",
    line_width=1.5,
    annotation_text=f"Start: ${S0:.2f}",
    annotation_position="right"
)

# --- Slider ---
steps = []
for day in range(0, N_DAYS + 1, 5):   # every 5 days
    step = dict(
        method="relayout",
        args=[{"shapes": [dict(
            type="line",
            x0=day, x1=day,
            y0=0, y1=1,
            xref="x", yref="paper",
            line=dict(color="orange", width=2)
        )]}],
        label=str(day)
    )
    steps.append(step)

sliders = [dict(
    active=0,
    steps=steps,
    currentvalue=dict(
        prefix="Trading day: ",
        font=dict(size=14)
    ),
    pad=dict(t=50)
)]

# --- Layout ---
fig.update_layout(
    title=dict(text="RKLB Monte Carlo simulation — 1,000 price paths (1 year)", x=0.5),
    xaxis_title="Trading day",
    yaxis_title="Price (USD)",
    template="plotly_dark",
    sliders=sliders,
    height=600,
    yaxis=dict(range=[0, np.percentile(final_prices, 97)]),
    xaxis=dict(range=[0, N_DAYS])
)

fig.show()

In [ ]:
import plotly.graph_objects as go

final_prices = price_paths[-1]
low_threshold = np.percentile(final_prices, 10)
high_threshold = np.percentile(final_prices, 90)

# --- Assign colours based on final outcome ---
colors = []
for i in range(N_SIMULATIONS):
    if final_prices[i] < low_threshold:
        colors.append("rgba(220, 53, 53, 0.15)")
    elif final_prices[i] > high_threshold:
        colors.append("rgba(40, 167, 69, 0.15)")
    else:
        colors.append("rgba(100, 149, 237, 0.10)")

# --- Build frames every 5 days ---
frames = []
slider_steps = []

for day in range(5, N_DAYS + 1, 5):
    frame_traces = []

    # Add each simulation path trimmed to current day
    for i in range(N_SIMULATIONS):
        frame_traces.append(go.Scatter(
            y=price_paths[:day, i],
            mode="lines",
            line=dict(color=colors[i], width=0.8),
            showlegend=False,
            hovertemplate="Day %{x}<br>Price: $%{y:.2f}<extra></extra>"
        ))

    # Add median path trimmed to current day
    frame_traces.append(go.Scatter(
        y=np.median(price_paths[:day, :], axis=1),
        mode="lines",
        line=dict(color="white", width=2.5, dash="dash"),
        name="Median",
        showlegend=True,
        hovertemplate="Day %{x}<br>Median: $%{y:.2f}<extra></extra>"
    ))

    frames.append(go.Frame(data=frame_traces, name=str(day)))

    slider_steps.append(dict(
        args=[[str(day)], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
        label=str(day),
        method="animate"
    ))

# --- Initial trace (day 5) ---
init_traces = []
for i in range(N_SIMULATIONS):
    init_traces.append(go.Scatter(
        y=price_paths[:5, i],
        mode="lines",
        line=dict(color=colors[i], width=0.8),
        showlegend=False
    ))
init_traces.append(go.Scatter(
    y=np.median(price_paths[:5, :], axis=1),
    mode="lines",
    line=dict(color="white", width=2.5, dash="dash"),
    name="Median"
))

# --- Build figure ---
fig = go.Figure(data=init_traces, frames=frames)

fig.add_hline(
    y=S0,
    line_dash="dot",
    line_color="yellow",
    line_width=1.5,
    annotation_text=f"Start: ${S0:.2f}",
    annotation_position="right"
)

fig.update_layout(
    title=dict(text="RKLB Monte Carlo simulation — 1,000 price paths", x=0.5),
    xaxis_title="Trading day",
    yaxis_title="Price (USD)",
    template="plotly_dark",
    height=600,
    yaxis=dict(range=[0, np.percentile(final_prices, 95)]),
    xaxis=dict(range=[0, N_DAYS]),
    sliders=[dict(
        active=0,
        steps=slider_steps,
        currentvalue=dict(prefix="Trading day: ", font=dict(size=14)),
        pad=dict(t=50)
    )]
)

fig.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- End prices from all simulations ---
end_prices = price_paths[-1]
returns = (end_prices - S0) / S0  # percentage return for each path

# --- VaR and CVaR at 95% and 99% confidence ---
VaR_95 = np.percentile(returns, 5)
VaR_99 = np.percentile(returns, 1)
CVaR_95 = returns[returns <= VaR_95].mean()
CVaR_99 = returns[returns <= VaR_99].mean()

print("=" * 45)
print(f"  RKLB 1-Year Risk Metrics (Monte Carlo)")
print("=" * 45)
print(f"  Starting price:       ${S0:.2f}")
print(f"  Median end price:     ${np.median(end_prices):.2f}")
print()
print(f"  95% VaR:    {VaR_95:.1%}  (${S0 * (1 + VaR_95):.2f})")
print(f"  99% VaR:    {VaR_99:.1%}  (${S0 * (1 + VaR_99):.2f})")
print()
print(f"  95% CVaR:   {CVaR_95:.1%}  (${S0 * (1 + CVaR_95):.2f})")
print(f"  99% CVaR:   {CVaR_99:.1%}  (${S0 * (1 + CVaR_99):.2f})")
print("=" * 45)

# --- Plot distribution of end returns ---
fig, ax = plt.subplots(figsize=(10, 5))

# Colour the histogram by zone
n, bins, patches = ax.hist(returns, bins=80, edgecolor="none")

for patch, left_edge in zip(patches, bins[:-1]):
    if left_edge < VaR_99:
        patch.set_facecolor("#dc3545")       # dark red — worst 1%
    elif left_edge < VaR_95:
        patch.set_facecolor("#fd7e14")       # orange — between 1% and 5%
    else:
        patch.set_facecolor("#4a9edd")       # blue — normal outcomes

# --- VaR lines ---
ax.axvline(VaR_95, color="orange", linewidth=2, linestyle="--",
           label=f"95% VaR: {VaR_95:.1%}")
ax.axvline(VaR_99, color="red", linewidth=2, linestyle="--",
           label=f"99% VaR: {VaR_99:.1%}")
ax.axvline(CVaR_95, color="orange", linewidth=2, linestyle=":",
           label=f"95% CVaR: {CVaR_95:.1%}")
ax.axvline(CVaR_99, color="red", linewidth=2, linestyle=":",
           label=f"99% CVaR: {CVaR_99:.1%}")
ax.axvline(0, color="white", linewidth=1, linestyle="-", alpha=0.4)

# --- Labels ---
ax.set_title("RKLB — Distribution of simulated 1-year returns", fontsize=14)
ax.set_xlabel("1-year return")
ax.set_ylabel("Number of simulations")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.legend(fontsize=10)
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#1a1a2e")
ax.tick_params(colors="white")
ax.xaxis.label.set_color("white")
ax.yaxis.label.set_color("white")
ax.title.set_color("white")
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# --- Sample 200 paths for performance ---
n_display = 200
indices = np.random.choice(N_SIMULATIONS, n_display, replace=False)
sampled_paths = price_paths[:, indices]

# --- Sort by final price so surface flows cleanly ---
sort_order = np.argsort(sampled_paths[-1, :])
Z = sampled_paths[:, sort_order].T

days = np.arange(N_DAYS)
sim_nums = np.arange(n_display)

colorscale = [
    [0.0,  "rgb(180, 30, 30)"],
    [0.3,  "rgb(220, 100, 30)"],
    [0.5,  "rgb(50, 100, 200)"],
    [0.75, "rgb(30, 180, 100)"],
    [1.0,  "rgb(200, 255, 200)"],
]

fig = go.Figure(data=[go.Surface(
    x=days,
    y=sim_nums,
    z=Z,
    colorscale=colorscale,
    opacity=0.85,
    contours=dict(
        x=dict(show=True, color="rgba(255,255,255,0.05)", width=1),
        y=dict(show=False),
        z=dict(show=False)
    ),
    showscale=True,
    colorbar=dict(
        title="End price ($)",
        tickfont=dict(color="white")
    ),
    hovertemplate="Day: %{x}<br>Simulation: %{y}<br>Price: $%{z:.2f}<extra></extra>"
)])

fig.update_layout(
    title=dict(
        text="RKLB GBM — 3D price path surface",
        x=0.5,
        font=dict(color="white", size=16)
    ),
    scene=dict(
        xaxis=dict(
            title="Trading day",
            backgroundcolor="#0d0d1a",
            gridcolor="rgba(255,255,255,0.1)",
            tickfont=dict(color="white")
        ),
        yaxis=dict(
            title="Simulation (ranked by outcome)",
            backgroundcolor="#0d0d1a",
            gridcolor="rgba(255,255,255,0.1)",
            tickfont=dict(color="white")
        ),
        zaxis=dict(
            title="Price (USD)",
            backgroundcolor="#0d0d1a",
            gridcolor="rgba(255,255,255,0.1)",
            tickfont=dict(color="white")
        ),
        bgcolor="#0d0d1a",
        camera=dict(
            eye=dict(x=1.8, y=-1.8, z=0.8)
        )
    ),
    paper_bgcolor="#0d0d1a",
    height=700,
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

# RKLB Monte Carlo Simulation
Monte Carlo simulation of Rocket Lab (RKLB) stock using Geometric Brownian Motion.
Includes interactive fan chart, 3D price surface, and VaR/CVaR risk metrics.